In [2]:
# import cplex

# # Initialize a CPLEX object
# c = cplex.Cplex()

# # Print the version
# print("CPLEX Version:", c.get_version())

## Smooth 1D manifolds

### Generate manifolds

In [3]:
from smooth_manifolds_generation.imagenet import init_imagenet
from smooth_manifolds_generation.network import NetworkType
from smooth_manifolds_generation.generation import OneDimensionalManifoldGenerator, GenerationConfig

imagenet_state = init_imagenet()

config = GenerationConfig(
    network_type=NetworkType.ALEXNET,   # ALEXNET, RESNET50, or VGG16
    range_factor=0.5,
    n_objects=8,
    n_samples=15,
)
generator = OneDimensionalManifoldGenerator(config, imagenet_state)

# Distributed generation (mirrors the MATLAB loop over id=1:28)
for batch_id in range(1, 29):
    generator.generate(batch_id=batch_id)

# Collect all batches into a single tuning-function array
tuning_function = generator.collect(batch_ids=range(1, 29))

Objects are 48 x 48 in the middle of 64 x 64 image
Loading ImageNet thumbnails from smooth_manifolds_generation/imagenet_all_thumbnails_64px.mat …
Skipped 0 images to choose 8 images with unique categories
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run001.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run002.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run003.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run004.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run005.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run006.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run007.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run008.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run009.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run010.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run011.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15_run012.npz
  Skipping existing: gen1d_alexnet_range0.5_P8_M15

### Direct estimation of classification capacity

In [4]:
from smooth_manifolds_analysis.capacity_analysis import (
    OneDimensionalCapacityAnalysis, CapacityAnalysisConfig,
)

cfg = CapacityAnalysisConfig(n_objects=128, range_factor=0.5, n_samples=15)
analysis = OneDimensionalCapacityAnalysis(cfg)
results = analysis.run(tuning_function, layer_number=20)  # 20 = feature layer for AlexNet
print(results)   # LayerCapacityResults with capacity_alpha_c per layer

  layer_number=20 → using 'output' (available: ['output'])
  [1/7] direction=x-translation  layer=output
 1000 neurons 15 conditions 8 objects


/Users/songdeying/Dropbox/Mac/Desktop/learning/neuro/object_manifolds/library/separability.py:159: UserWarning: cplex failed: CPLEX Error  1014: Parameter value too small.
  warnings.warn(f"cplex failed: {msg}")


  N=100 0.00 separable (20 samples)
  N=200 0.00 separable (20 samples)
  N=300 0.00 separable (20 samples)
  N=400 0.00 separable (20 samples)
  N=500 0.00 separable (20 samples)
  N=600 0.00 separable (20 samples)
  N=700 0.00 separable (20 samples)
  N=800 0.00 separable (20 samples)
  N=900 0.00 separable (20 samples)
  N=1000 0.00 separable (20 samples)
 Critical N not found
  [2/7] direction=y-translation  layer=output
 1000 neurons 15 conditions 8 objects
  N=100 0.00 separable (20 samples)
  N=200 0.00 separable (20 samples)
  N=300 0.00 separable (20 samples)
  N=400 0.00 separable (20 samples)
  N=500 0.00 separable (20 samples)
  N=600 0.00 separable (20 samples)
  N=700 0.00 separable (20 samples)
  N=800 0.00 separable (20 samples)
  N=900 0.00 separable (20 samples)
  N=1000 0.00 separable (20 samples)
 Critical N not found
  [3/7] direction=x-scale  layer=output
 1000 neurons 15 conditions 8 objects
  N=100 0.00 separable (20 samples)
  N=200 0.00 separable (20 samples)


### MFT geometry estimation

In [5]:
# from smooth_manifolds_analysis.low_rank_analysis import (
#     CovarianceLowRankAnalysis, LowRankAnalysisConfig,
# )

# cfg = LowRankAnalysisConfig(n_objects=128, range_factor=0.5, n_samples=15, max_k=5)
# analysis = CovarianceLowRankAnalysis(cfg)
# results = analysis.run(tuning_function, layer_number=20)
# # results.theory_capacity, results.mean_radius, results.mean_dimension

## Smooth 2-d manifolds

### Generate manifolds

In [6]:
from smooth_manifolds_generation.generation import RandomManifoldGenerator, GenerationConfig

config = GenerationConfig(
    network_type=NetworkType.ALEXNET,
    range_factor=0.5,
    n_objects=64,
    n_samples=201,
    n_transform_dims=2,
    n_batches=4,
)
generator = RandomManifoldGenerator(config)

for batch_id in range(1, config.n_batches * 2 + 1):
    generator.generate(batch_id=batch_id)

tuning_function = generator.collect(batch_ids=range(1, config.n_batches * 2 + 1))

  Skipping existing: genrand_alexnet_range0.5_P64_M201_run001.npz
  Skipping existing: genrand_alexnet_range0.5_P64_M201_run002.npz
  Skipping existing: genrand_alexnet_range0.5_P64_M201_run003.npz
  Skipping existing: genrand_alexnet_range0.5_P64_M201_run004.npz
  Skipping existing: genrand_alexnet_range0.5_P64_M201_run005.npz
  Skipping existing: genrand_alexnet_range0.5_P64_M201_run006.npz
  Skipping existing: genrand_alexnet_range0.5_P64_M201_run007.npz
  Skipping existing: genrand_alexnet_range0.5_P64_M201_run008.npz


### Direct capacity estimation

In [7]:
from smooth_manifolds_analysis.capacity_analysis import (
    RandomChangeCapacityAnalysis, CapacityAnalysisConfig,
)

cfg = CapacityAnalysisConfig(n_objects=64, range_factor=0.5, n_samples=201, n_transform_dims=2)
results = RandomChangeCapacityAnalysis(cfg).run(tuning_function, layer_number=20)

  transform_realization 1/2  layer=output
 1000 neurons 201 conditions 64 objects
  N=100 0.00 separable (20 samples)
  N=200 0.00 separable (20 samples)
  N=300 0.00 separable (20 samples)
  N=400 0.00 separable (20 samples)
  N=500 0.00 separable (20 samples)
  N=600 0.00 separable (20 samples)
  N=700 0.00 separable (20 samples)
  N=800 0.00 separable (20 samples)
  N=900 0.00 separable (20 samples)
  N=1000 0.00 separable (20 samples)
 Critical N not found
  transform_realization 2/2  layer=output
 1000 neurons 201 conditions 64 objects
  N=100 0.00 separable (20 samples)
  N=200 0.00 separable (20 samples)
  N=300 0.00 separable (20 samples)
  N=400 0.00 separable (20 samples)
  N=500 0.00 separable (20 samples)
  N=600 0.00 separable (20 samples)
  N=700 0.00 separable (20 samples)
  N=800 0.00 separable (20 samples)
  N=900 0.00 separable (20 samples)
  N=1000 0.00 separable (20 samples)
 Critical N not found


### MFT geometry estimation

In [ ]:
cfg = LowRankAnalysisConfig(n_objects=64, range_factor=0.5, n_samples=201, max_k=5, n_transform_dims=2)
results = CovarianceLowRankAnalysis(cfg).run(tuning_function, layer_number=20)

In [8]:
# import h5py

# imagenet_thumbnails = h5py.File('smooth_manifolds_generation/imagenet_all_thumbnails_64px.mat', 'r')

In [9]:
# print(imagenet_thumbnails.keys())
# print(imagenet_thumbnails['labels'])
# print(imagenet_thumbnails['thumbnails'])
# print(imagenet_thumbnails['#refs#'])

# labels = imagenet_thumbnails['labels'][:]
# thumbnails = imagenet_thumbnails['thumbnails'][:]
# # refs = imagenet_thumbnails['#refs#'][:]

# print(labels.shape)
# print(thumbnails.shape)
# # print(refs.shape)